# AutoGen · Brightwater's Analysis Team

Brightwater Outfitters sells outdoor gear online. The commercial team wants three numbers for Monday's review, and the order export they have is the kind every analyst gets: mostly right, with two things wrong in it that nobody mentions.

- **A re-export appended 70 orders a second time** — all West, all in Q3. Same order IDs, same everything.
- **A faulty batch of tents went to North in Q3**, and most of them came back.

The job is a small team that answers data questions by **writing and running code**, in one conversation: a **planner** says how to answer, an **analyst** writes pandas and runs it, a **reviewer** checks the method and the figures, and a **reporter** writes the answer up. That shape — agents talking to each other in a shared chat, one of them executing code — is what AutoGen is built around.

| | The question | The truth | What the export says as it stands |
|---|---|---|---|
| **Q1** | which region grew net revenue most, Q2 → Q3, and by how much? | **South, +37.6%** | West, +77.6% — the duplicates |
| **Q2** | which category had the highest return rate in Q3? | **tents, 19.4%** | tents, 17.6% |
| **Q3** | did the Trailhead promotion lift tent units ordered per day (returns included), and by how much? | **yes, +45.5%** | yes, +88.6% — the duplicates again |

Built twice, with the same people, prompts, routing rules and stopping rules: **with AutoGen** (`analysis_team.py`), as a group chat, and **from this repo's parts** (`team_scratch.py`), as a case file — chapter 6's shared board, agents and forms, chapter 3's tools.

| File | What it is |
|---|---|
| `brightwater.py` | the company: the export with its two planted problems, the questions, the true answers, the `Finding` form, the code gate |
| `analysis_team.py` | the team in AutoGen |
| `team_scratch.py` | the team from this repo's parts |
| `a_stubs.py` | stand-ins for every model call — the code they write still really runs |
| `a_checks.py` | what building it found out about AutoGen, each reproducible |
| `test_analysis.py` | offline tests for all of it |

---
# Part 1 · Build

## The team

Four AutoGen agents. Three are `AssistantAgent`s — a name, a model client, a system message, and a **description**, which is what the selector reads when it has to choose. The analyst is a `CodeExecutorAgent`: given a model client it asks the model for code, finds the ```` ```python ```` blocks in the reply, runs them, and reports what they printed. The reporter's `output_content_type=Finding` makes its reply structured output.

Every agent gets `TokenLimitedChatCompletionContext(client, 4500)` — as much of the recent chat as fits in 4,500 tokens, rather than the whole of it — because a group chat's history only grows, and Groq's free tier refuses any single request above 8,000 tokens a minute. A count of messages was tried first, and failed: one message carrying a program's output can be thousands of tokens on its own.

```python
# analysis_team.py
def build_team(client, work_dir, max_messages=MAX_MESSAGES, max_tokens=AUTOGEN_BUDGET, seconds=600, label="autogen ?",
               skills=False):
    from autogen_agentchat.agents import AssistantAgent, CodeExecutorAgent
    from autogen_agentchat.conditions import (FunctionalTermination, MaxMessageTermination,
                                              SourceMatchTermination, TimeoutTermination, TokenUsageTermination)
    from autogen_agentchat.teams import SelectorGroupChat
    from autogen_core.model_context import TokenLimitedChatCompletionContext
    from autogen_core.tools import FunctionTool

    # every agent sees as much of the chat as fits in 4,500 tokens, newest first: Groq's free tier
    # refuses any single request above 8,000 tokens a minute, and a group chat's history only grows.
    # The first version kept the last 10 MESSAGES -- one of which was a long program and its printout,
    # and the request came to 8,770 tokens. A count of messages says nothing about their size.
    recent = lambda: TokenLimitedChatCompletionContext(client, token_limit=4500)
    # With `skills`, the planner may read the team's procedures, through an ordinary AutoGen tool. Not
    # AutoGen's `memory=`: that puts memory into the context for the agent (all of it, with `ListMemory`),
    # and the point here is whether the agent picks the right procedures itself.
    tools = ([FunctionTool(B.read_skill, name="read_skill", description=B.READ_SKILL_DESCRIPTION)]
             if skills else [])
    planner = AssistantAgent("planner", client, system_message=PLANNER_WITH_SKILLS if skills else PLANNER,
                             model_context=recent(), tools=tools, reflect_on_tool_use=bool(skills),
                             max_tool_iterations=3,
                             description="Plans how to answer a data question, including checks on the data.")
    executor = _executor_class()(work_dir=work_dir, timeout=60, cleanup_temp_files=False)   # keep what ran
    executor.label = label
    analyst = _analyst_class()("analyst", executor, model_client=client,
                                system_message=ANALYST, model_context=recent(), approval_func=approve,
                                max_retries_on_error=2, supported_languages=["python"],
                                description="Writes and runs pandas code against orders.csv.")
    reviewer = AssistantAgent("reviewer", client, system_message=REVIEWER, model_context=recent(),
                              output_content_type=B.Review,
                              description="Checks the code and its output; approves or lists what to fix.")
    reporter = AssistantAgent("reporter", client, system_message=REPORTER, model_context=recent(),
                              output_content_type=B.Finding,
                              description="Writes the final answer once the reviewer has approved.")
    seen = []

    def second_rejection(delta):
        # a person takes over after the second review that does not approve: one revision, not a loop
        seen.extend(delta)
        return rejections(seen) >= MAX_REJECTIONS

    stop = (SourceMatchTermination(["reporter"]) | MaxMessageTermination(max_messages)
            | TokenUsageTermination(max_total_token=max_tokens) | TimeoutTermination(seconds)
            | FunctionalTermination(second_rejection))
    # a structured reply is a message type the team has to be told about, or it refuses to pass it on
    from autogen_agentchat.messages import StructuredMessage
    return SelectorGroupChat([planner, analyst, reviewer, reporter], model_client=client,
                             selector_func=next_speaker, termination_condition=stop,
                             custom_message_types=[StructuredMessage[B.Finding], StructuredMessage[B.Review]])
```

## Who speaks next

`SelectorGroupChat` normally has a model read the conversation and pick the next speaker. That is right for a judgement and wasteful for a rule, so `selector_func` decides every turn that is a rule: the question goes to the planner, any analysis to the reviewer, and the review — a form, below — says itself who goes next. The selector model is left with one case: a reviewer that answered in prose instead of filling in its form. The rules are a plain function, shared with the scratch build.

```python
# analysis_team.py
def route(source, content):
    """Who speaks after `source`. Shared with the scratch build, so both teams are routed by the same rules.

    A review is a `Review` form, so who fixes what is a field the reviewer filled in, not a word to look
    for in its prose. None only if the reviewer answered in prose, when the selector model reads it.
    """
    if source == "user":
        return "planner"
    if source == "planner":
        return "analyst"
    if source == "analyst":
        return "reviewer"
    if source == "reviewer":
        if isinstance(content, B.Review):
            return "reporter" if content.approved else ("planner" if content.fix_by == "planner" else "analyst")
        return None
    return None

def next_speaker(messages):
    """AutoGen's `selector_func`: the routing rules, applied to the last message in the chat."""
    last = messages[-1]
    return route(last.source, getattr(last, "content", ""))
```

## Evidence, and a review that says what to fix

The first live runs looped: the reviewer rejected, the analyst re-ran, the reviewer rejected again. Two causes, both engineering rather than judgement:

- **The reviewer was checking figures it could not see.** In AutoGen the analyst's code and its output are *events*, which other agents never receive — the reviewer read only the analyst's summary of them, and the scratch team's reviewer said so in as many words: *"no verifiable evidence was provided"*. So the analyst's report now carries, by code, the program that ran and what it printed. The same move as chapter 3's `verified()`: check the artifact, not the report about it.
- **"Not approved" was a paragraph.** The review is now a `Review` form — approved, the problems, who must fix them — so the routing reads a field and the analyst gets a list.

And one revision only: a second review that does not approve hands the case to a person, rather than going round again.

```python
# analysis_team.py
def with_evidence(report, code, output):
    """The analyst's report, with the code that ran and what it printed appended -- by code, every time.

    Added after both builds' reviewers kept rejecting: they were asked to check figures they could not
    see. In AutoGen the code and its output are EVENTS, which other agents never receive; the reviewer
    read only the analyst's summary of them, and the scratch build's reviewer said so in as many words:
    "no verifiable evidence was provided". A check needs the artifact, not a claim about it -- the same
    reason chapter 3's `verified()` looks at the world rather than at what the tool reported.
    """
    printed = output if len(output) <= EVIDENCE_LIMIT else output[:EVIDENCE_LIMIT] + "\n... (cut)"
    return f"{report}\n\nThe code that ran:\n```python\n{code}\n```\nWhat it printed:\n```\n{printed}\n```"

def _analyst_class():
    from autogen_agentchat.agents import CodeExecutorAgent
    from autogen_agentchat.base import Response
    from autogen_agentchat.messages import CodeExecutionEvent, CodeGenerationEvent, TextMessage

    class EvidenceAnalyst(CodeExecutorAgent):
        """AutoGen's code-executing agent, whose final report carries the code it ran and what it printed."""

        async def on_messages_stream(self, messages, cancellation_token):
            code = output = None
            async for item in super().on_messages_stream(messages, cancellation_token):
                if isinstance(item, CodeGenerationEvent) and item.code_blocks:
                    code = "\n\n".join(block.code for block in item.code_blocks)
                elif isinstance(item, CodeExecutionEvent):
                    output = item.result.output
                elif isinstance(item, Response) and code is not None and isinstance(item.chat_message, TextMessage):
                    report = item.chat_message
                    item = Response(chat_message=TextMessage(source=report.source, models_usage=report.models_usage,
                                                             content=with_evidence(report.content, code, output or "")),
                                    inner_messages=item.inner_messages)
                yield item

    return EvidenceAnalyst
```

```python
# brightwater.py
class Review(BaseModel):
    """The reviewer's verdict, as fields the routing can act on rather than a word to search for."""

    approved: bool = Field(description="True only if the code and its output, shown in the analyst's report, "
                                       "answer the question as it is defined.")
    problems: list[str] = Field(default_factory=list,
                                description="Each problem, specific enough to act on: what is wrong in the code or "
                                            "output, and what must be done about it. Empty if approved.")
    fix_by: Literal["analyst", "planner", "nobody"] = Field(
        description="analyst for a calculation or a data problem, planner for a wrong method, nobody if approved.")

    def render(self):
        if self.approved:
            return "APPROVED"
        return f"NOT APPROVED -- for the {self.fix_by}:\n" + "\n".join(f"- {p}" for p in self.problems)
```

The chat stops on whichever comes first — the reporter has spoken, the second rejection, 12 messages, a token budget, ten minutes. The budgets are per question and set from measured runs: 20,000 tokens for the AutoGen team, whose approved runs took 8–12k and whose one revision round, with the evidence in every review, takes about 19k; and 35,000 for the scratch team, whose turns each run a whole chapter 2 loop. Both limits came after a run with none: the scratch team spent about 100,000 tokens on one question, its reviewer rejecting the same analysis five times. A run that reaches a limit stops and says which. In AutoGen, termination conditions are objects combined with `|` (or `&`), which is in the `build_team` code above.

## Running model-written code

This is the part that needs care, because the analyst runs whatever the model writes, on this machine, as you. Three things were measured before any of it was trusted:

1. **AutoGen's local executor gives the code your whole environment.** It builds the child process's environment with `os.environ.copy()`, so a program the model wrote can read every API key the notebook loaded. Measured: `GROQ_API_KEY` visible. This subclass takes anything that looks like a secret out of the environment for exactly as long as the code runs:

```python
# analysis_team.py
def _executor_class():
    from autogen_ext.code_executors.local import LocalCommandLineCodeExecutor

    class Executor(LocalCommandLineCodeExecutor):
        """AutoGen's local executor, with secrets taken out of the environment while code runs.

        The parent copies `os.environ` into the child at the moment it starts it, so the secrets are
        removed for exactly that long and put back after. `os.environ` belongs to the whole process,
        which is why the lock: this is a workaround for a missing parameter, not a sandbox.
        """

        label = "autogen ?"

        async def execute_code_blocks(self, code_blocks, cancellation_token):
            with _ENV_LOCK:
                hidden = {k: os.environ.pop(k) for k in list(os.environ) if _SECRET.search(k)}
                try:
                    result = await super().execute_code_blocks(code_blocks, cancellation_token)
                finally:
                    os.environ.update(hidden)
            # what a program printed goes back into the conversation; cut it as chapter 3's invoke()
            # cuts a tool result. Uncut, one long printout pushed a request to 8,770 tokens and Groq
            # refused it (413: the free tier's limit is 8,000 tokens a minute)
            if len(result.output) > OUTPUT_LIMIT:
                result.output = result.output[:OUTPUT_LIMIT] + "\n... (output cut)"
            # one line per execution, so a run whose code keeps failing can be seen failing
            errors = [l for l in result.output.splitlines() if "Error" in l or "Traceback" in l]
            progress(*self.label.split(" ", 1), "-", f"code exit={result.exit_code} {errors[-1][:120] if errors else ''}")
            return result

    return Executor
```

2. **Code is checked before it runs**, through AutoGen's `approval_func`. The check is `brightwater.gate`: a list of patterns refusing code that reads environment variables, starts programs, uses the network, deletes files or builds code at run time. It is not a sandbox — Part 2 walks straight past it — which is why the environment is stripped as well. Without Docker there is no sandbox here at all.

```python
# brightwater.py
def gate(code):
    """(True, "") if the code may run; otherwise (False, why). Checked before every execution.

    A list of patterns, which a determined author gets past ("o" + "s"). It stops the accidents --
    a model reaching for an API key to "check the configuration", or deleting a file it made -- and
    says no out loud. It is not a sandbox; without Docker there is none here, which is why the
    executor is also given an environment with the secrets taken out.
    """
    for pattern, why in _FORBIDDEN.items():
        if re.search(pattern, code):
            return False, f"refused: the code {why}"
    return True, ""
```

```python
# analysis_team.py
def approve(request):
    """AutoGen calls this before running any code block. The decision is `brightwater.gate`'s."""
    from autogen_agentchat.agents import ApprovalResponse
    ok, why = B.gate(request.code)
    if not ok:
        REFUSED.append({"why": why, "code": request.code[:300]})
    return ApprovalResponse(approved=ok, reason=why or "allowed")
```

3. **The executor does not run in a Jupyter kernel on Windows.** It starts Python with `asyncio.create_subprocess_exec`, which a `SelectorEventLoop` does not implement — and ipykernel on Windows runs one. Measured: `NotImplementedError`. So each question gets a fresh team on its own thread, with a `ProactorEventLoop`: the same bridge chapter 7's `Session` built for MCP.

```python
# analysis_team.py
def run_team(question, client_factory=None, timeout=900, qid="?"):
    """Answer one question with a fresh team, on its own thread and event loop; return what happened.

    Everything asyncio -- the client, the team, the executor -- is created inside the thread's loop,
    because an async client belongs to the loop it was made in.
    """
    work_dir = tempfile.mkdtemp(prefix="brightwater-")
    B.write_orders(work_dir)
    box = {}

    async def go():
        from autogen_agentchat.base import TaskResult
        client = client_factory() if client_factory else model_client(label=f"autogen {qid}")
        team = build_team(client, work_dir, label=f"autogen {qid}")
        try:
            # streamed rather than `run`, so the messages so far survive a run that is stopped. The stream
            # is read to its end: returning from inside it abandons AutoGen's own shutdown task
            result = None
            async for item in team.run_stream(task=question):
                if isinstance(item, TaskResult):
                    result = item
                else:
                    box.setdefault("messages", []).append(item)
            return result
        finally:
            await client.close()

    def worker():
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        try:
            box["result"] = loop.run_until_complete(go())
        except Exception as failed:     # noqa: BLE001 -- reported, not raised
            box["error"] = f"{type(failed).__name__}: {failed}"
        finally:
            loop.close()

    began = time.monotonic()
    thread = threading.Thread(target=worker, daemon=True)
    thread.start()
    thread.join(timeout)
    seconds = round(time.monotonic() - began, 1)
    if thread.is_alive():
        return {"error": f"did not finish within {timeout}s", "seconds": seconds, "finding": None, "turns": [],
                "code_runs": 0, "speakers": [], "usage": {}, "stop_reason": None}
    if "error" in box:
        from types import SimpleNamespace
        partial = summarise(SimpleNamespace(messages=box.get("messages", []), stop_reason=None), seconds)
        return {**partial, "error": box["error"], "work_dir": work_dir}
    return summarise(box["result"], seconds)
```

## The same team, from scratch: a case file, not a chat

A group chat is AutoGen's design, and copying it here would mean copying its problems too: every agent reading a transcript that only grows, and evidence that has to be spliced into a message to reach the reviewer. This repo already has a different answer, chapter 6's **shared board** (`state.Shared`): one field per piece of work, each with **one owner** who may write it and a list of **readers** who see it. The program records each agent's work in its field, under the agent's name, so the ownership check applies to every write.

```python
# team_scratch.py
FIELDS = {"plan": "how to answer: the steps, and how to check the data first",
          "evidence": "the last program that ran, and what it printed",
          "report": "the analyst's account of what the program found",
          "review": "the reviewer's verdict, and anything that must be fixed",
          "finding": "the answer, as it goes to the commercial team"}
OWNER = {"plan": "planner", "evidence": "runner", "report": "analyst", "review": "reviewer", "finding": "reporter"}
READERS = {"plan": ["planner", "analyst", "reviewer"],
           "evidence": ["analyst", "reviewer", "reporter"],
           "report": ["reviewer", "reporter"],
           "review": ["planner", "analyst", "reviewer"],
           "finding": []}
```

Three things follow from that table, and each replaces something the AutoGen build had to add:

- **The evidence is a field the runner owns.** The `run_python` tool puts the program and what it printed on the board; the analyst could not write that field if it tried. The reviewer reads the evidence itself, so nothing like `with_evidence` and its subclass is needed.
- **Each agent reads only its fields.** The reporter never sees the plan or the review; the planner never sees the code. Nothing grows turn by turn, so there is no context window to manage.
- **`board.history` is the audit trail**: every write, by whom, from what, to what. It is what the run shows as its turns.

Running code is a **tool** rather than a separate executor agent: the analyst calls `run_python` with the program and reads what came back. Same gate as AutoGen; a subprocess started with a deliberately small environment — chapter 7's lesson that a process you launch starts with whatever you hand it.

```python
# team_scratch.py
def python_runner(work_dir, board, timeout=60, limit=AT.OUTPUT_LIMIT, label="scratch ?", refused=None):
    """A `run_python(code)` tool bound to one working folder. What ran, and what it printed, goes on the board."""
    def run_python(code: Annotated[str, "A complete Python program. It runs in a folder holding orders.csv; "
                                        "print everything you need to see."]) -> str:
        """Run a Python program and return what it printed (stdout and stderr)."""
        ok, why = B.gate(code)
        if not ok:
            if refused is not None:
                refused.append({"why": why, "code": code[:300]})
            return f"Not run -- {why}. Rewrite the program without it."
        script = Path(work_dir) / f"step_{len(list(Path(work_dir).glob('step_*.py')))}.py"
        script.write_text(code, encoding="utf-8")
        env = {k: v for k, v in os.environ.items() if k in _KEEP}
        env["PYTHONIOENCODING"] = "utf-8"
        try:
            done = subprocess.run([sys.executable, script.name], cwd=work_dir, env=env, capture_output=True,
                                  text=True, encoding="utf-8", timeout=timeout)
        except subprocess.TimeoutExpired:
            return f"Stopped after {timeout}s without finishing."
        output = (done.stdout + done.stderr).strip() or "(it printed nothing)"
        output = output[:limit] + ("\n... (cut)" if len(output) > limit else "")
        errors = [l for l in output.splitlines() if "Error" in l or "Traceback" in l]
        AT.progress(*label.split(" ", 1), "-", f"code exit={done.returncode} {errors[-1][:120] if errors else ''}")
        # the runner, not the analyst, puts the evidence on the case
        board.set("runner", "evidence", f"The program:\n```python\n{code}\n```\n"
                                        f"What it printed (exit {done.returncode}):\n```\n{output}\n```")
        return f"exit {done.returncode}:\n{output}"
    return run_python
```

A turn is one agent reading its view of the case and filling in its field. The analyst is an agent loop, because it has a tool to use. The reviewer and the reporter have none, so their forms go through chapter 1's `extract()`: the provider enforces the schema, so the reply *is* the form. A form still unusable after `extract()`'s retries comes back as chapter 6's **failure form**, and the case goes to a person. Who goes next is the same `route` as the AutoGen team.

```python
# team_scratch.py
def _ask(self, name, request, form=None):
    if self.answer is not None:
        return self.answer(name, request, form)
    if form is None:
        return self.agents[name].run(request)
    # The reviewer and the reporter have no tools: they read and fill in a form. An agent loop that
    # finishes by calling a `submit` tool would let the model answer in prose instead. Chapter 1's
    # `extract()` asks the provider to enforce the form's schema, so the reply IS the form, and validates
    # and retries behind that -- what AutoGen's `output_content_type` does.
    from failures import failure
    from llm import extract
    agent = self.agents[name]
    try:
        return extract(f"{agent.persona}\n\n{request}", form, model=agent.model)
    except ValueError as unusable:          # still not the form after extract's retries
        return failure(f"{name} did not fill in its form", str(unusable)[:200])

def _turn(self, name, question):
    """One person's turn: they read their view of the case, and what they produce goes in their field."""
    request = f"The question: {question}\n\n{self.board.view(name)}\n\nYour turn."
    form = FORMS.get(name)
    said = self._ask(name, request, form)
    if form is not None and not isinstance(said, form):
        # a failure form from `attempt`: it raised, was stopped, or answered in prose
        return None, f"needs a person: {said.answer} -- {said.unknowns}"[:300]
    text = said.render() if name == "reviewer" else said.model_dump_json() if name == "reporter" else said
    self.board.set(name, WRITES[name], text)
    return said, None

def _work(self, question):
    last, said = "user", question
    for _ in range(self.max_turns - 1):
        name = AT.route(last, said)
        said, failed = self._turn(name, question)
        if failed:
            return None, failed
        if name == "reporter":
            return said, grounded(said, self.board.values.get("evidence")) or "reporter answered"
        if name == "reviewer":
            self.reviews.append(said)
            if sum(not r.approved for r in self.reviews) >= AT.MAX_REJECTIONS:
                return None, "needs a person: the second review did not approve"
        last = name
    return None, "too many turns"
```

Before the finding goes out, one dull check in the spirit of chapter 6's `sendable`: its figure must be one the program printed. It catches a number the reporter made up; it cannot catch an analysis that printed the wrong one. The budget is chapter 6's `Meter` with a limit, checked before every model call:

```python
# team_scratch.py
def grounded(finding, evidence, tolerance=0.051):
    """Whether the finding's figure is one the program printed (to rounding). None if so, else why not.

    Chapter 6's `sendable`, for a Finding: a dull check, run before anything leaves the team. It
    catches a figure the reporter made up or worked out for itself; it cannot catch an analysis
    that printed the wrong figure.
    """
    printed = [float(n) for n in re.findall(r"-?\d+(?:\.\d+)?", evidence or "")]
    if any(abs(n - finding.value) <= tolerance or abs(n * 100 - finding.value) <= tolerance for n in printed):
        return None
    return f"held for a person: {finding.value} is not a figure the program printed"

class Budget(Meter):
    """Chapter 6's `Meter`, with a limit: every call counted and logged, none made past the limit.

    The scratch team's turns are whole chapter 2 loops -- the analyst's can be several calls -- so a
    check between turns is not enough, for the same reason it was not enough for AutoGen.
    `over` says why it stopped, because a form turn reports the stop as its own failure.

        with Budget(35_000, "scratch Q1") as budget:
            team.run(question)
        budget.summary()
    """

    def __init__(self, limit=AT.SCRATCH_BUDGET, label="scratch ?"):
        self.limit, self.label, self.over = limit, label, None

    def spent(self):
        return sum(call["prompt"] + call["completion"] for call in self.calls)

    def __enter__(self):
        import llm
        super().__enter__()
        metered = llm.client.chat.completions.create

        def create(*args, **kwargs):
            if self.spent() >= self.limit:
                self.over = f"token budget of {self.limit:,} reached ({self.spent():,} spent)"
                AT.progress(*self.label.split(" ", 1), self.spent(), "BUDGET REACHED -- stopping")
                raise AT.BudgetExceeded(self.over)
            response = metered(*args, **kwargs)
            AT.progress(*self.label.split(" ", 1), self.spent())
            return response

        llm.client.chat.completions.create = create
        return self
```

## Procedures as skills

Q1 failed in both builds because nobody checked the export for repeated orders, though AutoGen's Q2 analyst did exactly that unprompted. A team that has done this work before has *procedures*: chapter 4's procedural memory, the "how-to" an agent picks up and reuses. Here they are nine short procedures, written generically. None names this export or its problems, and only three bear on Q1. The rest are there so that what gets measured is whether the planner **picks** the right ones from their descriptions, not whether it obeys a single note it was handed.

```python
# brightwater.py
SKILLS = {
    "deduplicate-records": (
        "any question that totals, counts, averages or compares the rows of an export or a log",
        "1. Find the column that identifies a record (an id such as order_id).\n"
        "2. Count the ids that appear more than once, and print that count.\n"
        "3. If the repeated rows are identical, keep one of each and say how many were dropped.\n"
        "4. If they differ, do not guess which is right: report them."),
    "net-revenue": (
        "a question about revenue, sales value or takings",
        "1. Revenue per row = units x unit_price x (1 - discount).\n"
        "2. Check discount is a fraction between 0 and 1 before using it.\n"
        "3. Apply the question's definition of which rows count (returned orders, cancelled orders) exactly "
        "as stated, and say what was excluded."),
    "period-growth": (
        "comparing one figure between two periods (month on month, quarter on quarter)",
        "1. Compute the figure for each period from the same cleaned data.\n"
        "2. Growth % = (later / earlier - 1) x 100.\n"
        "3. Print both periods' figures as well as the growth, for every group compared."),
    "rate-per-day": (
        "an average per day over a date window",
        "1. Count the days in the window inclusively, including days with no rows.\n"
        "2. Divide the window's total by that count."),
    "currency-conversion": (
        "amounts recorded in more than one currency",
        "1. Find each row's currency.\n2. Convert with the rate for the transaction date, not today's.\n"
        "3. Report in one currency, and say which."),
    "outlier-check": (
        "a numeric column that may hold values far outside the usual range",
        "1. Compute the quartiles and the interquartile range (IQR).\n"
        "2. Flag values below Q1 - 1.5 x IQR or above Q3 + 1.5 x IQR.\n3. Report them; drop none without a reason."),
    "cohort-retention": (
        "how many customers come back after their first purchase",
        "1. Group customers by the month of their first order.\n"
        "2. For each later month, count the share of each group that ordered again."),
    "significance-test": (
        "deciding whether a difference between two groups is more than chance",
        "1. State the two groups and the measure.\n2. Use a two-proportion z-test for rates, Welch's t-test for "
        "means.\n3. Report the p-value next to the difference, not instead of it."),
    "timezone-normalisation": (
        "timestamps recorded in more than one time zone",
        "1. Convert every timestamp to UTC before grouping by day.\n2. Say which zone the report's days are in."),
}
RELEVANT = {"Q1": {"deduplicate-records", "net-revenue", "period-growth"}}

def read_skill(name: str) -> str:
    """The full text of one of the team's procedures."""
    if name.strip() not in SKILLS:
        return f"No procedure named {name!r}. The procedures are:\n{skill_catalogue()}"
    when, steps = SKILLS[name.strip()]
    return f"{name} -- for {when}.\n{steps}"
```

The planner sees the catalogue (names and when each applies) in the tool's description and reads procedures in full only when it decides to. In AutoGen that is an ordinary `FunctionTool` on the planner, not AutoGen's `memory=`, which puts memory into an agent's context for it (all of it, with `ListMemory`), so there would be nothing to pick. The analyst, a `CodeExecutorAgent`, takes neither tools nor memory, so the planner is where procedures can enter, and the analyst follows the plan. The scratch build gives its planner the same text through a chapter 3 `Tool`:

```python
# analysis_team.py
PLANNER_WITH_SKILLS = PLANNER + """

The team keeps the procedures it has learned; the read_skill tool lists them. Before you plan, read the ones that
apply to this question, and build their steps into your plan."""
```

```python
# team_scratch.py
def read_skill(self, name: Annotated[str, "The procedure's name, exactly as listed"]) -> str:
    """The full text of one of the team's procedures."""
    self.skills_read.append(name)
    return B.read_skill(name)
```

---
# Part 2 · Run and show

In [1]:
import warnings

warnings.filterwarnings("ignore")

import a_checks as K
import a_stubs as S
import analysis_team as AT
import brightwater as B
import team_scratch as TS
autogen, scratch = {}, {}

## 1 · Offline

Every model call replaced by a stand-in — AutoGen's own `ReplayChatCompletionClient` for the AutoGen team. The code the stand-ins write is a careful analysis, and **it really runs**, through each build's executor, against the real export:

In [2]:
for qid in B.QUESTIONS:
    offline = AT.run_team(B.QUESTIONS[qid], client_factory=lambda q=qid: S.replay(q), timeout=120)
    team = TS.Team()
    team.answer = S.scratch(team, qid)
    by_hand = team.run(B.QUESTIONS[qid])
    print(f"{qid}  AutoGen: {B.grade(qid, offline['finding'])['right']}, speakers {offline['speakers']}")
    print(f"    scratch: {B.grade(qid, by_hand['finding'])['right']}, speakers {by_hand['speakers']}")

Q1  AutoGen: True, speakers ['user', 'planner', 'analyst', 'analyst']
    scratch: True, speakers ['planner', 'analyst', 'reviewer', 'reporter']


Q2  AutoGen: True, speakers ['user', 'planner', 'analyst', 'analyst']
    scratch: True, speakers ['planner', 'analyst', 'reviewer', 'reporter']


Q3  AutoGen: True, speakers ['user', 'planner', 'analyst', 'analyst']
    scratch: True, speakers ['planner', 'analyst', 'reviewer', 'reporter']


What the reviewer is shown on the scratch team, from the last offline case: its own fields and nothing else.

In [3]:
print(team.board.view("reviewer"))

What the team has established so far:
- plan: 1. Check the data for duplicates. 2. Compute what the question defines.  (set by planner)
- evidence: The program:
```python
import pandas as pd
df = pd.read_csv("orders.csv").drop_duplicates("order_id")
t = df[df.category == "tents"]
during = t[(t.order_date >= "2026-07-01") & (t.order_date <= "2026-07-14")].units.sum() / 14
before = t[(t.order_date >= "2026-06-17") & (t.order_date <= "2026-06-30")].units.sum() / 14
print("per day before:", round(before, 2), "during:", round(during, 2), "lift %:", round((during / before - 1) * 100, 1))
```
What it printed (exit 0):
```
per day before: 3.14 during: 4.57 lift %: 45.5
```  (set by runner)
- report: The figures are in what the program printed.  (set by analyst)
- review: APPROVED  (set by reviewer)


## 2 · AutoGen, live

One fresh team per question. Each run shows what every agent said and what the code printed.

In [3]:
autogen["Q1"] = AT.run_team(B.QUESTIONS["Q1"], qid="Q1")
K.save("autogen", "Q1", autogen["Q1"])
K.show_run("Q1", autogen["Q1"])
print("\nstop:", autogen["Q1"]["stop_reason"], "|", autogen["Q1"]["usage"], "|", autogen["Q1"]["seconds"], "s")

Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


--- planner: **Steps to determine the region with the biggest net‑revenue growth from Q2 → Q3 2026** 1. **Load & preview the file** - Open `orders.csv` and view the first/last rows. - Verify column names, data types (dates, numeric fields), and that the file contains rows for 2026. 2. **Validate the data** - **Date sanity:** Ensure...
--- analyst (code ran): exit 1: The script ran, then exited with an error (POSIX exit code: 1) Its output was: Traceback (most recent call last): File "C:\Users\91879\AppData\Local\Temp\brightwater-6fg5vd01\tmp_code_294e1e711fc1bad6920da2a4870c9f7be3f59cc61d2ccd357f0c14a2902bde72.py", line 76, in <module> df_valid['...
--- analyst (code ran): exit 0: C:\Users\91879\AppData\Local\Temp\brightwater-6fg5vd01\tmp_code_9649e9776655100b714ffddd5d78fd2c44564fdb59a7618bb73c8dc971aa39dc.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Q2' has dtype incompatible with float64, plea

In [4]:
autogen["Q2"] = AT.run_team(B.QUESTIONS["Q2"], qid="Q2")
K.save("autogen", "Q2", autogen["Q2"])
K.show_run("Q2", autogen["Q2"])
print("\nstop:", autogen["Q2"]["stop_reason"], "|", autogen["Q2"]["usage"], "|", autogen["Q2"]["seconds"], "s")

Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


--- planner: **Steps to answer the question (no code needed):** 1. **Load & preview the file** - Open *orders.csv* in a spreadsheet or data‑viewer. - Verify that all expected columns are present (`order_id`, `order_date`, `category`, `returned`, …) and that the first few rows look sensible (no shifted columns, proper headers). 2. *...
--- analyst (code ran): exit 0: Duplicate order_id values found: ['BW-12396', 'BW-12788', 'BW-13070', 'BW-11784', 'BW-11785', 'BW-11865', 'BW-11938', 'BW-12124', 'BW-12226', 'BW-12380', 'BW-12440', 'BW-12742', 'BW-12950', 'BW-13026', 'BW-13384', 'BW-11822', 'BW-12080', 'BW-12438', 'BW-13420', 'BW-12020', 'BW-12064', 'BW-12142', 'BW-12482', 'B...
--- analyst: tents 19.40 The code that ran: ```python import pandas as pd import sys # Load the data try: df = pd.read_csv('orders.csv') except FileNotFoundError: print("orders.csv not found") sys.exit(1) # ---------- Data validation ---------- suspicious = False # 1. Check required columns required_cols = {'order

In [5]:
autogen["Q3"] = AT.run_team(B.QUESTIONS["Q3"], qid="Q3")
K.save("autogen", "Q3", autogen["Q3"])
K.show_run("Q3", autogen["Q3"])
print("\nstop:", autogen["Q3"]["stop_reason"], "|", autogen["Q3"]["usage"], "|", autogen["Q3"]["seconds"], "s")

Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


...

Error processing publish message for reporter_fbb2108e-5e52-4646-90d8-6a8373b77a2c/fbb2108e-5e52-4646-90d8-6a8373b77a2c
Traceback (most recent call last):
  File "C:\Users\91879\anaconda3\envs\agentic-forge\Lib\site-packages\autogen_core\_single_threaded_agent_runtime.py", line 606, in _on_message
    return await agent.on_message(
           ^^^^^^^^^^^^^^^^^^^^^^^
    ...<2 lines>...
    )
    ^
  File "C:\Users\91879\anaconda3\envs\agentic-forge\Lib\site-packages\autogen_core\_base_agent.py", line 119, in on_message
    return await self.on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\91879\anaconda3\envs\agentic-forge\Lib\site-packages\autogen_agentchat\teams\_group_chat\_sequential_routed_agent.py", line 67, in on_message_impl
    return await super().on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\91879\anaconda3\envs\agentic-forge\Lib\site-packages\autogen_core\_routed_agent.

Error processing publish message for planner_fbb2108e-5e52-4646-90d8-6a8373b77a2c/fbb2108e-5e52-4646-90d8-6a8373b77a2c
Traceback (most recent call last):
  File "C:\Users\91879\anaconda3\envs\agentic-forge\Lib\site-packages\autogen_core\_single_threaded_agent_runtime.py", line 606, in _on_message
    return await agent.on_message(
           ^^^^^^^^^^^^^^^^^^^^^^^
    ...<2 lines>...
    )
    ^
  File "C:\Users\91879\anaconda3\envs\agentic-forge\Lib\site-packages\autogen_core\_base_agent.py", line 119, in on_message
    return await self.on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\91879\anaconda3\envs\agentic-forge\Lib\site-packages\autogen_agentchat\teams\_group_chat\_sequential_routed_agent.py", line 72, in on_message_impl
    return await super().on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\91879\anaconda3\envs\agentic-forge\Lib\site-packages\autogen_core\_routed_agent.p

Error processing publish message for analyst_fbb2108e-5e52-4646-90d8-6a8373b77a2c/fbb2108e-5e52-4646-90d8-6a8373b77a2c
Traceback (most recent call last):
  File "C:\Users\91879\anaconda3\envs\agentic-forge\Lib\site-packages\autogen_core\_single_threaded_agent_runtime.py", line 606, in _on_message
    return await agent.on_message(
           ^^^^^^^^^^^^^^^^^^^^^^^
    ...<2 lines>...
    )
    ^
  File "C:\Users\91879\anaconda3\envs\agentic-forge\Lib\site-packages\autogen_core\_base_agent.py", line 119, in on_message
    return await self.on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\91879\anaconda3\envs\agentic-forge\Lib\site-packages\autogen_agentchat\teams\_group_chat\_sequential_routed_agent.py", line 72, in on_message_impl
    return await super().on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\91879\anaconda3\envs\agentic-forge\Lib\site-packages\autogen_core\_routed_agent.p

Error processing publish message for reviewer_fbb2108e-5e52-4646-90d8-6a8373b77a2c/fbb2108e-5e52-4646-90d8-6a8373b77a2c
Traceback (most recent call last):
  File "C:\Users\91879\anaconda3\envs\agentic-forge\Lib\site-packages\autogen_core\_single_threaded_agent_runtime.py", line 606, in _on_message
    return await agent.on_message(
           ^^^^^^^^^^^^^^^^^^^^^^^
    ...<2 lines>...
    )
    ^
  File "C:\Users\91879\anaconda3\envs\agentic-forge\Lib\site-packages\autogen_core\_base_agent.py", line 119, in on_message
    return await self.on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\91879\anaconda3\envs\agentic-forge\Lib\site-packages\autogen_agentchat\teams\_group_chat\_sequential_routed_agent.py", line 72, in on_message_impl
    return await super().on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\91879\anaconda3\envs\agentic-forge\Lib\site-packages\autogen_core\_routed_agent.

error: RuntimeError: BudgetExceeded: token budget of 20,000 reached (24,565 spent)
Traceback:
Traceback (most recent call last):

  File "C:\Users\91879\anaconda3\envs\agentic-forge\Lib\site-packages\autogen_agentchat\teams\_group_chat\_chat_agent_container.py", line 133, in handle_request
    async for msg in self._agent.on_messages_stream(self._message_buffer, ctx.cancellation_token):
    ...<4 lines>...
            await self._log_message(msg)

  File "C:\Users\91879\anaconda3\envs\agentic-forge\Lib\site-packages\autogen_agentchat\agents\_assistant_agent.py", line 953, in on_messages_stream
    async for inference_output in self._call_llm(
    ...<15 lines>...
            yield inference_output

  File "C:\Users\91879\anaconda3\envs\agentic-forge\Lib\site-packages\autogen_agentchat\agents\_assistant_agent.py", line 1109, in _call_llm
    model_result = await model_client.create(
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<4 lines>...
    )
    ^

  File "C:\Users\91879\One

## 3 · From scratch, live

The same questions, people, prompts and rules, as a case file on chapter 6's board.

In [2]:
scratch["Q1"] = TS.Team(qid="Q1").run(B.QUESTIONS["Q1"], qid="Q1")
K.save("scratch", "Q1", scratch["Q1"])
K.show_run("Q1", scratch["Q1"])
print("\nstop:", scratch["Q1"]["stop_reason"], "|", scratch["Q1"]["usage"])

--- planner: **Steps to determine the region with the highest net‑revenue growth from Q2 → Q3 2026** 1. **Load & inspect the file** – Open *orders.csv* and verify that all expected columns are present (`order_id, order_date, region, channel, category, product, units, unit_price, discount, returned`). Check data types (e.g., dates p...
--- runner (code ran): The program: ```python import pandas as pd, sys, numpy as np # Load CSV try: df = pd.read_csv('orders.csv') except Exception as e: print('Error loading CSV:', e) sys.exit() # Print columns print('Columns:', list(df.columns)) # Basic inspection print('Row count:', len(df)) # Check for missing critical columns required =...
--- analyst: West, 77.63%
--- reviewer: APPROVED
--- reporter: {"entity":"West","value":77.6,"answer":"West grew net revenue the most from Q2 to Q3 2026, with a 77.6% increase. This growth was calculated using net revenue = units × unit_price × (1 - discount) after excluding returned orders. All data fields passed 


WRONG: answered West 77.6, the truth is South 37.6

stop: reporter answered | {'calls': 5, 'prompt': 6802, 'completion': 2489, 'total': 9291, 'seconds': 40}


In [2]:
scratch["Q2"] = TS.Team(qid="Q2").run(B.QUESTIONS["Q2"], qid="Q2")
K.save("scratch", "Q2", scratch["Q2"])
K.show_run("Q2", scratch["Q2"])
print("\nstop:", scratch["Q2"]["stop_reason"], "|", scratch["Q2"]["usage"])

--- planner: **Plan – Steps to answer the question** 1. **Load the file** `orders.csv` into a data‑analysis environment (e.g., pandas, Excel). 2. **Validate the schema** – confirm that all expected columns (`order_id, order_date, region, channel, category, product, units, unit_price, discount, returned`) are present and correctly t...
--- runner (code ran): The program: ```python import pandas as pd, sys, json # Load data df = pd.read_csv('orders.csv') # Print schema check expected_cols = ['order_id','order_date','region','channel','category','product','units','unit_price','discount','returned'] missing = [c for c in expected_cols if c not in df.columns] if missing: print...
--- analyst: tents 0.17647058823529413
--- reviewer: NOT APPROVED -- for the planner: - The current aggregation counts each row, including 70 duplicate order_id rows, which may double‑count orders and distort return rates; the analysis must decide whether to deduplicate by order_id or treat each line‑item as a sepa


RIGHT: answered tents 19.4, the truth is tents 19.4

stop: reporter answered | {'calls': 9, 'prompt': 13349, 'completion': 5310, 'total': 18659, 'seconds': 85}


In [4]:
scratch["Q3"] = TS.Team(qid="Q3").run(B.QUESTIONS["Q3"], qid="Q3")
K.save("scratch", "Q3", scratch["Q3"])
K.show_run("Q3", scratch["Q3"])
print("\nstop:", scratch["Q3"]["stop_reason"], "|", scratch["Q3"]["usage"])

--- planner: **Plan – Steps to answer the question** 1. **Load & preview the data** – Open `orders.csv` and view the first few rows to confirm all expected columns are present and correctly named (`order_id, order_date, region, channel, category, product, units, unit_price, discount, returned`). 2. **Validate date field** – Ensure ...
--- runner (code ran): The program: ```python import pandas as pd, sys, numpy as np # Load data try: df = pd.read_csv('orders.csv') except Exception as e: print('Error loading CSV:', e) sys.exit() # Initial inspection print('Columns:', list(df.columns)) # Show first few rows print('Head:', df.head(3).to_dict(orient='records')) # Check for mi...
--- runner (code ran): The program: ```python import pandas as pd, numpy as np, sys # Load try: df = pd.read_csv('orders.csv') except Exception as e: print('Error loading CSV:', e) sys.exit() # Basic info print('Rows:', len(df)) # Drop duplicate order_id, keep first if df['order_id'].duplicated().any(): dup = df['o

## 4 · What it took to run code safely

**The event loop.** AutoGen's code executor under the loop a script runs, and the one a Jupyter kernel runs on Windows:

In [2]:
for loop, result in K.event_loops().items():
    print(f"{loop}: {result}")

ProactorEventLoop (a script's default): ran: hello
SelectorEventLoop (what a Jupyter kernel runs on Windows): NotImplementedError: 


**What model-written code can read**, through AutoGen's executor and through this project's:

In [3]:
for executor, seen in K.secrets().items():
    print(f"{executor}: {seen}")

AutoGen's executor -- the code can see: ['GEMINI_API_KEY', 'GOOGLE_API_KEY', 'GROQ_API_KEY']
this project's executor -- the code can see: []


**The gate**, on five programs — the last one reads the same key as the second:

In [4]:
for program, verdict in K.gate_examples().items():
    print(f"{program:36} {verdict}")

reads the CSV                        allowed
prints an API key                    refused: the code reads environment variables
calls a web service                  refused: the code uses the network
deletes a file                       refused: the code deletes files
the same key, written differently    allowed


**Termination**: a reviewer that never approves, against a replayed model, with the message limit lowered to 12:

In [5]:
print(K.termination())

{'messages': 10, 'stop_reason': 'Functional termination condition met', 'speakers': ['user', 'planner', 'analyst', 'analyst']}


## 5 · Side by side

In [2]:
import ast
import io
import tokenize
from pathlib import Path


def code_lines(path):
    source = Path(path).read_text(encoding="utf-8")
    docstrings = set()
    for node in ast.walk(ast.parse(source)):
        body = getattr(node, "body", None)
        if isinstance(body, list) and body and isinstance(body[0], ast.Expr) \
                and isinstance(getattr(body[0], "value", None), ast.Constant) and isinstance(body[0].value.value, str):
            docstrings.update(range(body[0].lineno, body[0].end_lineno + 1))
    lines = set()
    for token in tokenize.generate_tokens(io.StringIO(source).readline):
        if token.type not in (tokenize.COMMENT, tokenize.NL, tokenize.NEWLINE, tokenize.INDENT,
                              tokenize.DEDENT, tokenize.ENDMARKER):
            lines.update(l for l in range(token.start[0], token.end[0] + 1) if l not in docstrings)
    return len(lines)


runs = K.load()                              # every live run above, as saved
print(f"{'':6}{'AutoGen':>26}{'from scratch':>30}")
for qid in B.QUESTIONS:
    a, s = B.grade(qid, runs[("autogen", qid)]["finding"]), B.grade(qid, runs[("scratch", qid)]["finding"])
    verdict = lambda g: "right" if g["right"] else "WRONG" if g["entity"] else "none"    # none: stopped, or held for a person
    print(f"{qid:6}{a['entity'] or '-':>8} {a['value'] or 0:>6} {verdict(a):>6}   "
          f"{s['entity'] or '-':>10} {s['value'] or 0:>6} {verdict(s):>6}"
          f"     truth: {a['want'][0]} {a['want'][1]}")
for build in ("autogen", "scratch"):
    mine = [r for (b, _), r in runs.items() if b == build]
    print(f"{build:8} tokens {sum(r['usage'].get('total', 0) for r in mine):>7,}  "
          f"calls {sum(r['usage'].get('calls', 0) for r in mine):>4}  turns {[r['turns'] for r in mine]}")
on_messages = sum((r.get("usage on messages") or {}).get("total", 0) for (b, _), r in runs.items() if b == "autogen")
print(f"{'':8} AutoGen's tokens as its own messages report them: {on_messages:,}")
print(f"lines of code: AutoGen {code_lines('analysis_team.py')}, scratch {code_lines('team_scratch.py')}")

                         AutoGen                  from scratch


Q1        West   77.6  WRONG         West   77.6  WRONG     truth: South 37.6
Q2       tents   19.4  right        tents   19.4  right     truth: tents 19.4
Q3           -      0   none          yes   45.5  right     truth: yes 45.5
autogen  tokens  56,624  calls   19  turns [10, 7, 10]
scratch  tokens  44,350  calls   20  turns [6, 10, 7]
         AutoGen's tokens as its own messages report them: 36,682


lines of code: AutoGen 250, scratch 167


## 6 · Procedures as skills, on Q1

The question both builds got wrong, again, with the procedures available to the planner. What to read: which procedures the planner opened, against the three Q1 needs; whether the analysis then looked for repeated rows; and the answer.

In [2]:
skilled = {}
skilled["scratch"] = TS.Team(qid="Q1", skills=True).run(B.QUESTIONS["Q1"], qid="Q1")
K.save("scratch-skills", "Q1", skilled["scratch"])
K.show_run("Q1", skilled["scratch"])
print("\nstop:", skilled["scratch"]["stop_reason"], "|", skilled["scratch"]["usage"])
print(K.skills_report("Q1", skilled["scratch"]))

--- planner: **Plan (numbered steps)** 1. **Load & inspect the export** - Verify that all required columns (`order_date`, `region`, `units`, `unit_price`, `discount`, `returned`) are present and correctly typed (dates as dates, numeric fields as numbers). - Spot‑check a few rows to confirm `discount` is expressed as a fraction (0 –...
--- runner (code ran): The program: ```python import pandas as pd, sys, numpy as np, os, json, textwrap # Load CSV try: df = pd.read_csv('orders.csv') except Exception as e: print('Error loading orders.csv:', e) sys.exit() # Required columns required = ['order_date', 'region', 'units', 'unit_price', 'discount', 'returned'] missing = [c for c...
--- analyst: South 37.57%
--- reviewer: APPROVED
--- reporter: {"entity":"South","value":37.6,"answer":"South experienced the strongest net revenue growth between Q2 and Q3 2026. Its revenue increased by 37.6% compared to the previous quarter. This makes South the top‑performing region for that period.","data_issue


RIGHT: answered South 37.6, the truth is South 37.6

stop: reporter answered | {'calls': 7, 'prompt': 9283, 'completion': 3774, 'total': 13057, 'seconds': 41}
{'read': ['net-revenue', 'period-growth'], 'needed and read': ['net-revenue', 'period-growth'], 'needed, not read': ['deduplicate-records'], 'read, not needed': [], 'the analysis checked for duplicates': True}


In [3]:
skilled["autogen"] = AT.run_team(B.QUESTIONS["Q1"], qid="Q1", skills=True, budget=AT.SKILLS_BUDGET)
K.save("autogen-skills", "Q1", skilled["autogen"])
K.show_run("Q1", skilled["autogen"])
print("\nstop:", skilled["autogen"]["stop_reason"], "|", skilled["autogen"]["usage"])
print(K.skills_report("Q1", skilled["autogen"]))

Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


...

Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


Model openai/gpt-oss-120b not found. Using cl100k_base encoding.


--- planner: [FunctionCall(id='fc_7886eca4-02a3-4052-a178-88e4abf1de9b', arguments='{"name":"net-revenue"}', name='read_skill')]
--- planner: [FunctionExecutionResult(content="net-revenue -- for a question about revenue, sales value or takings.\n1. Revenue per row = units x unit_price x (1 - discount).\n2. Check discount is a fraction between 0 and 1 before using it.\n3. Apply the question's definition of which rows count (returned orders, cancelled orders) ...
--- planner: [FunctionCall(id='fc_2477a203-c992-4f15-8bb7-9700f631da9c', arguments='{"name":"period-growth"}', name='read_skill')]
--- planner: [FunctionExecutionResult(content="period-growth -- for comparing one figure between two periods (month on month, quarter on quarter).\n1. Compute the figure for each period from the same cleaned data.\n2. Growth % = (later / earlier - 1) x 100.\n3. Print both periods' figures as well as the growth, for every group comp...
--- planner: [FunctionCall(id='fc_3c89741c-01d3-470f-9378-c97554e9


RIGHT: answered South 37.6, the truth is South 37.6

stop: 'reporter' answered | {'calls': 11, 'prompt': 17418, 'completion': 7388, 'total': 24806}
{'read': ['net-revenue', 'period-growth', 'deduplicate-records'], 'needed and read': ['deduplicate-records', 'net-revenue', 'period-growth'], 'needed, not read': [], 'read, not needed': [], 'the analysis checked for duplicates': True}


In [2]:
runs = K.load()
print(f"{'Q1':22}{'answer':>14}{'checked duplicates':>20}{'procedures read':>18}{'tokens':>9}")
for build, label in [("autogen", "AutoGen"), ("autogen-skills", "AutoGen + skills"),
                     ("scratch", "scratch"), ("scratch-skills", "scratch + skills")]:
    if (build, "Q1") not in runs:
        continue
    r = runs[(build, "Q1")]
    g = B.grade("Q1", r["finding"])
    answer = f"{g['entity'] or '-'} {g['value'] or ''}".strip()
    read = ", ".join(r.get("skills read") or []) or "-"
    print(f"{label:22}{answer:>14}{str(r.get('checked duplicates', '-')):>20}   {read:<55}{r['usage'].get('total', 0):>9,}")
print(f"\nneeded for Q1: {', '.join(sorted(B.RELEVANT['Q1']))};  truth: {B.truth()['Q1']['entity']} {B.truth()['Q1']['value']}")

Q1                            answer  checked duplicates   procedures read   tokens


AutoGen                    West 77.6                   -   -                                                         21,856
AutoGen + skills          South 37.6                True   net-revenue, period-growth, deduplicate-records           24,806
scratch                    West 77.6                   -   -                                                          9,291
scratch + skills          South 37.6                True   net-revenue, period-growth                                13,057

needed for Q1: deduplicate-records, net-revenue, period-growth;  truth: South 37.6


## Findings

**Without procedures, both builds got Q1 wrong the same way; with them, both got it right.** Read these as cases, not rates.

| | AutoGen | From scratch |
|---|---|---|
| **Q1** | West +77.6%, approved: duplicates kept | West +77.6%, approved: duplicates kept |
| **Q1, with procedures** | **South +37.6%, right**, 24,806 tokens | **South +37.6%, right**, 13,057 tokens |
| **Q2** | **tents 19.4%, right** | **tents 19.4%, right**, after the reviewer sent the method back over 70 duplicate rows |
| **Q3** | the right figure, 45.45%, approved after the reviewer caught a real bug, then **stopped at the budget** before the reporter could write it up | **yes, +45.5%, right**, approved first time |
| tokens, billed, Q1–Q3 | 56,624 in 19 calls | 44,350 in 20 calls |

**Q1 is the export's trap, and without help no plumbing caught it.** The 70 duplicated West orders were never dropped, and both reviewers approved: a reviewer that sees the code and its printout can only catch what the code reveals. Both write-ups also described data problems that were not there or not handled. AutoGen's said discount percentages had been converted; there were none. The scratch build's said there were no data issues, with 70 duplicates in the file. The reported figure is checked against what the program printed (`grounded`); the `data_issues` text is not, and a reader would trust it most.

**Procedures as skills.** Nine generic procedures, three relevant to Q1, offered to the planner as a catalogue it reads from (section 6):
- **AutoGen's planner picked exactly the three that apply**, and none of the six distractors, and built the dedupe steps into its plan. Its analyst dropped the duplicates, and its reviewer sent the work back for one thing more: the analyst had *reported* the count wrongly (140, both copies of each, instead of 70).
- **The scratch planner never opened `deduplicate-records`**, yet its plan said "De-duplicate on `order_id` if duplicates exist". The one-line description in the catalogue was enough. So "did it pick the procedure" and "did the procedure help" are separate questions, and only the saved plan tells them apart.
- **How a procedure is described decides whether it is picked.** Both planners chose by the kind of question: the catalogue entries that matched were the ones written as "a question about revenue", "comparing one figure between two periods", "any question that totals, counts, averages or compares rows".
- **What it cost.** Q1 went from 21,856 tokens (wrong) to 24,806 (right) for AutoGen, and from 9,291 to 13,057 for the scratch build.

**What AutoGen gave**: a group chat with a pluggable speaker selector, termination conditions combined with `|`, a code-executing agent, tools on any `AssistantAgent`, structured output on the reviewer and reporter, streaming, and a **replay client** that made the whole team testable offline with the real executor running the code. Its `memory=` was not the right fit for procedures: it puts memory into an agent's context for it, so there is nothing for the agent to pick.

**What it took to run AutoGen safely**, all reproduced in section 4:
- **The budget cannot be a termination condition.** `TokenUsageTermination` is checked between turns and counts usage attached to messages; the analyst's turn is several calls, and calls whose messages the team never emits are not counted at all. AutoGen's messages report **36,682** tokens for what the API billed as **56,624**. The limit has to live in the model client, checked before every call.
- **The code executor passes on the whole environment.** Model-written code could read `GROQ_API_KEY`, `GEMINI_API_KEY` and `GOOGLE_API_KEY`; a subclass strips them while the code runs, and sets UTF-8 output, which Windows does not default to. The pattern gate in front of it is not a sandbox.
- **It does not run in a Jupyter kernel on Windows.** The executor needs a `ProactorEventLoop`; the kernel runs a `SelectorEventLoop`. Each team runs on its own thread, with its own loop.
- **Evidence has to be carried by hand.** The code and its output are events, which other agents never receive, so an `EvidenceAnalyst` subclass appends both to the analyst's report.
- **A budget that stops a run before its last step loses that step's work**, as Q3 shows: approved, and never written up.

**The scratch build is a different design, not a smaller AutoGen.** It is a case file on chapter 6's shared board: each agent reads only its own fields, and the evidence field is written by the code runner, which the analyst cannot write, so nothing like `EvidenceAnalyst` is needed. The reviewer and reporter fill in their forms through chapter 1's `extract()`, with the provider enforcing the schema, as AutoGen's `output_content_type` does. Prompts stay small without a context-window policy, which is most of the token difference. It is 167 lines against 250, on chapters 1, 2, 3 and 6, which this repo had already built. What it lacks: a code executor with a Docker option, streaming, and a selector that can be a model when the rules run out.